# 🧪 Laboratório 2 — Preços de congestionamento, expansão da rede e o gerador que mente

**Minicurso LaPSEE · UNESP Ilha Solteira**

Na aula vocês viram de onde saem os preços nodais. Hoje vocês são **a equipe de planejamento
da transmissão**: vão medir quanto custa o congestionamento em duas redes, defender uma
linha de expansão diante da diretoria, testar cenários *what-if* — e descobrir o que
acontece com tudo isso quando um gerador **mente sobre os próprios custos**.

## Regras do jogo

1. Pode trabalhar em equipes.
2. Os blocos `# ╔══ COMPLETAR` são de vocês; a **VERIFICAÇÃO** (`assert`) logo abaixo diz
   se o código está correto.
3. As perguntas 📝 são o **entregável**: ao final, vocês redigem um memorando de 1 página
   ao regulador.
4. Tentar não avançar com uma VERIFICAÇÃO em vermelho.



---
## 0 · Setup (entregue completo — apenas rodar)

Além dos auxiliares de sempre, este setup traz **o mercado do IEEE 39** que usaremos nos
blocos B–D. O desenho conecta com o Laboratório 1: a comunidade pequena da bipartição de
Fiedler (14 barras: 14–16, 18–23, 26, 32–35) vira um **bolsão importador**:

- a carga do bolsão é de **2.440 MW**, mas a geração local soma só **2.427 MW** — e é toda
  **cara** (\$45/MWh);
- o resto do sistema é **barato** (\$18/MWh), mas a energia barata precisa atravessar a rede
  para chegar lá;
- os limites térmicos originais do caso, apertados a 70%, fazem o transporte saturar.



In [ ]:
import json
import logging
import warnings

import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import pandapower as pp
import pandapower.networks as pn

RANDOM_SEED = 2026
logging.getLogger("pandapower").setLevel(logging.CRITICAL)
warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.figsize": (9, 5), "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False, "font.size": 11})
AZUL, VERMELHO, CINZA = "#1F3A5F", "#C8102E", "#a0a0a0"


def coords_das_barras(net):
    "{barra: (x, y)} — compatível com pandapower 2.x e 3.x."
    if hasattr(net, "bus_geodata") and len(getattr(net, "bus_geodata", [])):
        return {int(b): (float(net.bus_geodata.at[b, "x"]),
                         float(net.bus_geodata.at[b, "y"])) for b in net.bus.index}
    return {int(b): tuple(json.loads(g)["coordinates"][:2])
            for b, g in net.bus["geo"].items()}


def grafo_topologico(net):
    G = nx.Graph(); G.add_nodes_from(sorted(net.bus.index))
    for _, ln in net.line.iterrows():
        G.add_edge(int(ln.from_bus), int(ln.to_bus))
    for _, tr in net.trafo.iterrows():
        G.add_edge(int(tr.hv_bus), int(tr.lv_bus))
    return G


def foto_lmp(net):
    "Resumo de um OPF resolvido: LMPs, spread e restrições ativas (como na aula)."
    barras = sorted(net.bus.index)
    lmp = net.res_bus["lam_p"].reindex(barras)
    n_bind = int((net.res_line["loading_percent"] > 99.9).sum()) \
           + int((net.res_trafo["loading_percent"] > 99.9).sum())
    return {"lmp": lmp, "spread": float(lmp.max() - lmp.min()), "n_binding": n_bind,
            "custo": float(net.res_cost)}


def mapa_de_precos(net, titulo=""):
    "Rede pintada por LMP, ramos saturados em vermelho."
    pos = coords_das_barras(net); G = grafo_topologico(net)
    sat = set()
    for i, r in net.line.iterrows():
        if net.res_line.loading_percent[i] > 99.9:
            sat.add(frozenset((int(r.from_bus), int(r.to_bus))))
    for i, r in net.trafo.iterrows():
        if net.res_trafo.loading_percent[i] > 99.9:
            sat.add(frozenset((int(r.hv_bus), int(r.lv_bus))))
    barras = sorted(net.bus.index)
    lmp = net.res_bus["lam_p"].reindex(barras)
    fig, ax = plt.subplots(figsize=(10.5, 6.5))
    for u, v in G.edges():
        (x1, y1), (x2, y2) = pos[u], pos[v]
        cor, lw = (VERMELHO, 3.0) if frozenset((u, v)) in sat else ("lightgray", 1.0)
        ax.plot([x1, x2], [y1, y2], color=cor, lw=lw, zorder=2 if cor == VERMELHO else 1)
    sc = ax.scatter([pos[b][0] for b in barras], [pos[b][1] for b in barras],
                    c=lmp.values, cmap="plasma", s=210, edgecolors="k", zorder=3)
    for b in barras:
        ax.annotate(str(b), pos[b], ha="center", va="center", fontsize=6.5,
                    color="w", zorder=4)
    plt.colorbar(sc, label="LMP [$/MWh]")
    ax.set_title(titulo); ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
    plt.tight_layout(); plt.show()


# ---------- o mercado do IEEE 39 (blocos B–D) ----------
BOLSAO = [14, 15, 16, 18, 19, 20, 21, 22, 23, 26, 32, 33, 34, 35]   # comunidade de Fiedler
CUSTO_CARO, CUSTO_BARATO = 45.0, 18.0


def mercado_case39(esc_limites=0.7, ofertas_extra=None, fator_linha=None):
    '''IEEE 39 como mercado: bolsão importador caro vs resto barato.

    esc_limites : aperta os limites térmicos originais (0.7 = 70%).
    ofertas_extra : dict {barra_do_gerador: oferta} para SOBRESCREVER ofertas (bloco D).
    fator_linha : (índice_da_linha, fator) para expandir UMA linha (blocos B-C).
    '''
    net = pn.case39()
    net.line["max_i_ka"] = net.line["max_i_ka"] * esc_limites
    if fator_linha is not None:
        li, f = fator_linha
        net.line.loc[li, "max_i_ka"] *= f
    net.line["max_loading_percent"] = 100.0
    net.trafo["max_loading_percent"] = 100.0
    net.poly_cost.drop(net.poly_cost.index, inplace=True)      # zera os custos do caso
    barras_g = [int(b) for b in net.gen.bus] + [int(net.ext_grid.bus.iloc[0])]
    ofertas = {g: (CUSTO_CARO if g in BOLSAO else CUSTO_BARATO) for g in barras_g}
    if ofertas_extra:
        ofertas.update(ofertas_extra)
    pp.create_poly_cost(net, 0, "ext_grid",
                        cp1_eur_per_mw=ofertas[int(net.ext_grid.bus.iloc[0])])
    for gi, g in net.gen.iterrows():
        pp.create_poly_cost(net, gi, "gen", cp1_eur_per_mw=ofertas[int(g.bus)])
    return net


print("Setup pronto —", "pandapower", pp.__version__)

---
## Bloco A · Quanto custa o congestionamento?

### A.1 — O IEEE 57 vira mercado

O IEEE 57 já vem com **custos diferenciados** (geradores de \$20 e de \$40, com termo
quadrático) — mas com linhas infinitas (`max_i_ka` ≈ 50 kA!). A tabela de limites por nível
de tensão, calibrada para gerar congestionamento sem inviabilizar o despacho:

| kV | 500 | 345 | 230 | 161 | 138 | 115 |
|---|---|---|---|---|---|---|
| MVA | 720 | 400 | 200 | 120 | 96 | 80 |

Convertam para `max_i_ka` com a fórmula da aula, $I_{max} = S / (\sqrt{3}\,V)$, e resolvam o
despacho ótimo.

In [ ]:
MVA_POR_KV = {500: 720, 345: 400, 230: 200, 161: 120, 138: 96, 115: 80}
net57 = pn.case57()

# ╔══ COMPLETAR ══ dica: para cada linha, leiam vn_kv da barra from_bus, busquem o MVA na
# ║   tabela e gravem max_i_ka = MVA/(sqrt(3)*kV); depois max_loading_percent = 100 para
# ║   linhas e trafos; por fim pp.rundcopp(net57)
# ✏️  ESCREVAM O CÓDIGO DE VOCÊS AQUI (apaguem a linha do raise ao terminar)
raise NotImplementedError("Bloco a completar — vejam a dica acima")

# ── VERIFICAÇÃO automática ───────────────────────────────────────────────────
foto57 = foto_lmp(net57)
assert 1 <= foto57["n_binding"] <= 3, f"esperava 1-3 restrições ativas, deu {foto57['n_binding']}"
assert 25 < foto57["spread"] < 45, f"spread fora da faixa esperada: {foto57['spread']:.1f}"
print(f"✅ VERIFICAÇÃO OK — custo {foto57['custo']:,.0f} $/h | "
      f"spread {foto57['spread']:.1f} $/MWh | {foto57['n_binding']} ramos saturados")
mapa_de_precos(net57, "IEEE 57 como mercado — onde o preço salta, uma linha saturou")

### A.2 — A régua do planejador: o custo do congestionamento

Spread de preço impressiona, mas a régua de quem decide é outra:

$$C_{cong} \;=\; C(\text{rede com limites}) \;-\; C(\text{"placa de cobre"})$$

onde a *placa de cobre* é a mesma rede com limites infinitos (a energia mais barata sempre
chega). $C_{cong}$ é, por hora, **quanto o país paga a mais por a rede ser finita** — e o
teto teórico do que vale investir em transmissão.

Calculem $C_{cong}$ para as duas redes do dia (o mercado do IEEE 39 sai de
`mercado_case39()`, já pronto no setup).

In [ ]:
# ╔══ COMPLETAR ══ dica: a "placa de cobre" é a mesma rede com max_i_ka = 999 em todas as
# ║   linhas e novo rundcopp. C_cong = custo_com_limites - custo_placa.
# ║   Para o IEEE 39: net39 = mercado_case39() (com limites) e outra instância para a placa.
# ✏️  ESCREVAM O CÓDIGO DE VOCÊS AQUI (apaguem a linha do raise ao terminar)
raise NotImplementedError("Bloco a completar — vejam a dica acima")

# ── VERIFICAÇÃO automática ───────────────────────────────────────────────────
assert 1000 < ccong57 < 2500, f"C_cong do IEEE 57 fora da faixa: {ccong57:,.0f}"
assert 15000 < ccong39 < 25000, f"C_cong do IEEE 39 fora da faixa: {ccong39:,.0f}"
comp = pd.DataFrame({
    "rede": ["IEEE 57", "IEEE 39 (mercado)"],
    "custo com limites [$/h]": [foto57["custo"], foto39["custo"]],
    "C_cong [$/h]": [ccong57, ccong39],
    "C_cong [% do custo ideal]": [100*ccong57/float(n57_cu.res_cost),
                                  100*ccong39/float(n39_cu.res_cost)],
    "spread [$/MWh]": [foto57["spread"], foto39["spread"]],
    "ramos saturados": [foto57["n_binding"], foto39["n_binding"]],
}).round(1)
print("✅ VERIFICAÇÃO OK")
display(comp)
mapa_de_precos(net39, "IEEE 39 como mercado — o bolsão de Fiedler (Lab 1) virou bolsão de preço")

💡 Que pode comentar dos resultados?


📝 **Pergunta A:** anualizem o $C_{cong}$ do IEEE 39 (×8760 h). Se uma linha nova custa
~US\$ 50 milhões, o congestionamento de um ano "paga" quantas linhas? 

> *Resposta de vocês aqui:* ______



---
## Bloco B · Defender uma expansão diante da diretoria

A diretoria autoriza **uma** obra: aumentar em 50% a capacidade de **uma** linha do IEEE 39.
Qual escolher? A intuição diz "a mais carregada". Vamos testar a intuição.

### B.1 — O ranking what-if

Para cada uma das 8 linhas mais carregadas do caso base: criem o mercado com essa linha
expandida (`mercado_case39(fator_linha=(indice, 1.5))`), resolvam o OPF e registrem a
**economia** = custo base − custo com a expansão.

In [ ]:
custo_base = foto39["custo"]
top8 = net39.res_line.loading_percent.nlargest(8).index

# ╔══ COMPLETAR ══ dica: para cada li em top8, net_x = mercado_case39(fator_linha=(int(li), 1.5));
# ║   pp.rundcopp(net_x); economia = custo_base - net_x.res_cost. Guardem também
# ║   from_bus/to_bus e o loading base da linha.
# ✏️  ESCREVAM O CÓDIGO DE VOCÊS AQUI (apaguem a linha do raise ao terminar)
raise NotImplementedError("Bloco a completar — vejam a dica acima")

# ── VERIFICAÇÃO automática ───────────────────────────────────────────────────
assert len(ranking) == 8 and ranking["economia [$/h]"].max() > 5000, \
    "a melhor expansão deveria economizar bem mais de 5.000 $/h"
carregadas_inuteis = ranking[(ranking["loading base [%]"] > 85)
                             & (ranking["economia [$/h]"] < 100)]
assert len(carregadas_inuteis) >= 4, \
    "deveriam existir várias linhas MUITO carregadas cuja expansão vale ~zero"
print("✅ VERIFICAÇÃO OK")
display(ranking)
melhor = int(ranking.iloc[0]["linha"])
print(f"A linha vencedora é a {melhor} "
      f"({int(ranking.iloc[0]['de'])}→{int(ranking.iloc[0]['para'])}).")

💡 Que chama a atenção nos resultados obtidos?



### B.2 — Os perdedores da sua linha (código entregue)

Antes de ir à diretoria, uma pergunta que todo planejador esquece: **quem perde** com a
expansão? Comparem os LMPs antes e depois.

In [ ]:
net_melhor = mercado_case39(fator_linha=(melhor, 1.5))
pp.rundcopp(net_melhor)
dlmp = (net_melhor.res_bus.lam_p - net39.res_bus.lam_p).sort_values()

fig, ax = plt.subplots(figsize=(12, 4.2))
cores = [VERMELHO if v > 0.5 else AZUL if v < -0.5 else CINZA for v in dlmp.values]
ax.bar(dlmp.index.astype(str), dlmp.values, color=cores)
ax.axhline(0, color="k", lw=0.6)
ax.set_xlabel("barra"); ax.set_ylabel("ΔLMP [$/MWh]")
ax.set_title(f"Expandir a linha {melhor} em 50%: azul = preço cai · "
             "vermelho = preço SOBE (os perdedores)")
ax.tick_params(axis="x", labelsize=7)
plt.tight_layout(); plt.show()

sobem, caem = int((dlmp > 0.5).sum()), int((dlmp < -0.5).sum())
print(f"Barras com LMP em queda: {caem} | com LMP em ALTA: {sobem} "
      f"(até {dlmp.max():+.1f} $/MWh)")
print(f"Economia total do sistema: {custo_base - float(net_melhor.res_cost):,.0f} $/h")

💡 Que chama a atenção nos resultados obtidos?

📝 **Pergunta B:** redijam 3 frases para a diretoria defendendo a linha escolhida
(economia, % do $C_{cong}$ capturado) e 2 frases antecipando **quem** fará lobby contra e
**com que argumento**. Os geradores do bolsão apoiariam ou combateriam a obra?

> *Resposta de vocês aqui:* ______


📝 **Pergunta D:** vocês são o monitor de mercado. Proponham duas medidas para detectar/
mitigar esse comportamento e discutam: a expansão da linha — paga por todos — é um remédio eficiente para um problema de *conduta*?

> *Resposta de vocês aqui:* ______

---

In [ ]:
fatores = [1.0, 1.25, 1.5, 2.0, 3.0]

# ╔══ COMPLETAR ══ dica: para cada fator f: net_f = mercado_case39(fator_linha=(melhor, f));
# ║   rundcopp; economia = custo_base - res_cost; saturados via foto_lmp(net_f)["n_binding"]
# ✏️  ESCREVAM O CÓDIGO DE VOCÊS AQUI (apaguem a linha do raise ao terminar)
raise NotImplementedError("Bloco a completar — vejam a dica acima")

# ── VERIFICAÇÃO automática ───────────────────────────────────────────────────
eco = curva["economia [$/h]"].values
assert (np.diff(eco) >= -1e-6).all(), "relaxar um limite nunca pode PIORAR o ótimo"
assert curva["ramos saturados"].iloc[3] >= 3, \
    "ao dobrar a capacidade, novas linhas deveriam saturar (o congestionamento migra!)"
print("✅ VERIFICAÇÃO OK")
display(curva)

fig, ax1 = plt.subplots(figsize=(9, 4.6))
ax1.plot(curva["fator"], eco, "o-", color=AZUL, lw=2)
ax1.set_xlabel(f"fator de capacidade da linha {melhor}")
ax1.set_ylabel("economia vs. caso base [$/h]", color=AZUL)
ax2 = ax1.twinx()
ax2.bar(curva["fator"], curva["ramos saturados"], width=0.08, color=VERMELHO, alpha=0.45)
ax2.set_ylabel("ramos saturados", color=VERMELHO); ax2.grid(False)
ax1.set_title("Retornos decrescentes — e o congestionamento que MIGRA para outras linhas")
plt.tight_layout(); plt.show()

💡 Que chama a atenção nos resultados obtidos?



### C.2 — E a heurística "ligar os extremos de preço"? (código entregue)

No Laboratório 1, ligar os extremos do vetor de Fiedler era a melhor jogada para a coesão
($\lambda_2$). Tentemos o análogo de mercado: uma linha **nova** de 600 MVA entre a barra
mais barata e a mais cara.

In [ ]:
b_min, b_max = int(foto39["lmp"].idxmin()), int(foto39["lmp"].idxmax())
net_nova = mercado_case39()
pp.create_line_from_parameters(net_nova, b_min, b_max, length_km=1.0,
                               r_ohm_per_km=0.5, x_ohm_per_km=30.0, c_nf_per_km=0.0,
                               max_i_ka=600 / (np.sqrt(3) * 345))
net_nova.line["max_loading_percent"] = 100.0
pp.rundcopp(net_nova)
eco_nova = custo_base - float(net_nova.res_cost)
print(f"Linha NOVA {b_min}→{b_max} (600 MVA): economia = {eco_nova:,.0f} $/h")
print(f"Expandir a linha {melhor} em 50%      : economia = "
      f"{float(ranking.iloc[0]['economia [$/h]']):,.0f} $/h")


📝 **Pergunta C:** com a curva de C.1, proponham o fator de expansão que vocês de fato
recomendariam (não existe resposta única — argumentem com a economia marginal de cada salto
e com a migração do congestionamento).

> *Resposta de vocês aqui:* ______



---
## Bloco D · Informação imperfeita: o gerador que mente

Até aqui, todo mundo declarou o custo verdadeiro. Mas o OPF do operador despacha com as
**ofertas declaradas** — e o bolsão tem um segredo aritmético:

- carga do bolsão: **2.440 MW**;
- importação máxima (limites das linhas) + capacidade dos *outros* geradores locais:
  **não cobrem** a carga;
- ⇒ o gerador da barra **34** é **pivotal**: uma fatia da sua produção é *obrigatória*,
  seja qual for o preço que ele pedir.

O que acontece quando ele percebe isso?

### D.1 — O laboratório de mentiras

Completem a função `despachar(oferta_34)`: ela roda o mercado com a oferta declarada da
barra 34 e devolve as cinco grandezas que importam. A chave é o **custo real**: o despacho
sai das ofertas *declaradas*, mas a conta de combustível se paga com os custos
*verdadeiros* (\$45 no bolsão, \$18 fora).

In [ ]:
def despachar(oferta_34):
    "Roda o mercado com a barra 34 ofertando `oferta_34` e devolve o placar completo."
    net = mercado_case39(ofertas_extra={34: float(oferta_34)})
    pp.rundcopp(net)
    gi34 = int(net.gen[net.gen.bus == 34].index[0])

    # ╔══ COMPLETAR ══ dica: despacho_34 = res_gen.p_mw[gi34]; lmp_34 = res_bus.lam_p[34];
    # ║   lucro_34 = (lmp_34 - CUSTO_CARO) * despacho_34;
    # ║   custo_real = ext_grid·18 + soma de res_gen.p_mw × custo VERDADEIRO de cada gerador
    # ║   (45 se a barra está em BOLSAO, 18 caso contrário);
    # ║   pago_cargas = soma de LMP_barra × carga_da_barra
    # ✏️  ESCREVAM O CÓDIGO DE VOCÊS AQUI (apaguem a linha do raise ao terminar)
    raise NotImplementedError("Bloco a completar — vejam a dica acima")

    return {"oferta": float(oferta_34), "despacho_34": despacho_34, "lmp_34": lmp_34,
            "lucro_34": lucro_34, "custo_real": custo_real, "pago_cargas": pago_cargas}


# ── VERIFICAÇÃO automática ───────────────────────────────────────────────────
honesto = despachar(45.0)
assert abs(honesto["lucro_34"]) < 50, "ofertando o custo verdadeiro, o lucro marginal deve ser ≈ 0"
assert abs(honesto["custo_real"] - custo_base) < 5, "com todos honestos, custo real = custo do OPF"
print(f"✅ VERIFICAÇÃO OK — honesto: despacho {honesto['despacho_34']:.0f} MW, "
      f"LMP \\$45, lucro ≈ 0")

### D.2 — A varredura: mentir compensa? (código entregue)

In [ ]:
ofertas = [45, 50, 55, 60, 70, 80, 90]
placar = pd.DataFrame([despachar(o) for o in ofertas])

# ── VERIFICAÇÃO automática ───────────────────────────────────────────────────
piso = placar["despacho_34"].iloc[2:]
assert piso.max() - piso.min() < 2, "a partir de ~55, o despacho deveria travar num piso (pivotal!)"
assert (placar["custo_real"].max() - placar["custo_real"].min()) < 10, \
    "o custo REAL do sistema não deveria mudar com a mentira"
assert placar["lucro_34"].is_monotonic_increasing, "o lucro deveria crescer com a oferta"
print("✅ VERIFICAÇÃO OK")
display(placar.round(0))

fig, axes = plt.subplots(2, 2, figsize=(12, 7.5))
for ax, col, cor, titulo in [
        (axes[0, 0], "despacho_34", AZUL, "Despacho da barra 34 [MW] — trava num PISO"),
        (axes[0, 1], "lmp_34", VERMELHO, "LMP no bolsão [$/MWh] — segue a oferta 1:1"),
        (axes[1, 0], "lucro_34", VERMELHO, "Lucro do mentiroso [$/h]"),
        (axes[1, 1], "custo_real", CINZA, "Custo REAL do sistema [$/h] — não se move")]:
    ax.plot(placar["oferta"], placar[col], "o-", color=cor, lw=2)
    ax.set_xlabel("oferta declarada da barra 34 [$/MWh]"); ax.set_title(titulo, fontsize=10)
axes[1, 1].set_ylim(custo_base - 3000, custo_base + 3000)
plt.tight_layout(); plt.show()

print(f"Com oferta de \\$90: lucro de {placar['lucro_34'].iloc[-1]:,.0f} $/h para o gerador, "
      f"\ne as cargas pagam {placar['pago_cargas'].iloc[-1] - placar['pago_cargas'].iloc[0]:,.0f} $/h A MAIS"
      f" — com custo físico idêntico.")

💡 Que chama a atenção nos resultados obtidos?



### D.3 — O planejador enganado (código entregue)

Última volta do parafuso: o regulador **não conhece** os custos verdadeiros — ele mede o
congestionamento com as ofertas declaradas. O que a mentira faz com as decisões de expansão?

In [ ]:
def ccong_aparente(ofertas_extra=None):
    "Custo de congestionamento medido com as ofertas DECLARADAS."
    n1 = mercado_case39(ofertas_extra=ofertas_extra); pp.rundcopp(n1)
    n2 = mercado_case39(ofertas_extra=ofertas_extra); n2.line["max_i_ka"] = 999.0
    pp.rundcopp(n2)
    return float(n1.res_cost) - float(n2.res_cost)


def economia_aparente_linha(li, ofertas_extra=None):
    "Economia da expansão da linha `li` ×1.5, medida com as ofertas DECLARADAS."
    n1 = mercado_case39(ofertas_extra=ofertas_extra); pp.rundcopp(n1)
    n2 = mercado_case39(ofertas_extra=ofertas_extra, fator_linha=(li, 1.5)); pp.rundcopp(n2)
    return float(n1.res_cost) - float(n2.res_cost)


mentira = {34: 90.0}
resumo = pd.DataFrame({
    "métrica": ["custo de congestionamento [$/h]",
                f"economia aparente da linha {melhor} ×1.5 [$/h]"],
    "mundo honesto": [ccong_aparente(), economia_aparente_linha(melhor)],
    "com a mentira (oferta 34 = $90)": [ccong_aparente(mentira),
                                        economia_aparente_linha(melhor, mentira)],
}).round(0)
display(resumo)
fator_distorcao = resumo.iloc[0, 2] / resumo.iloc[0, 1]
print(f"O regulador enganado mede um congestionamento {fator_distorcao:.1f}× maior do que o real.")
print(f"(A economia REAL da linha, a custos verdadeiros, segue sendo ~7.800 $/h.)")

💡 Que chama a atenção nos resultados obtidos?


📝 **Pergunta D:** vocês são o monitor de mercado. Proponham duas medidas para detectar/
mitigar esse comportamento (ideias: comparar ofertas com custos auditados de combustível,
*bid caps* locais quando há fornecedor pivotal, contratos de longo prazo no bolsão) e
discutam: a expansão da linha — paga por todos — é um remédio eficiente para um problema de
*conduta*?

> *Resposta de vocês aqui:* ______
